# 125 - Build Pfam Benchmark Ground Truth

Generates the ground-truth data required by `123_pfam_subdomain_divergence_benchmark.ipynb`.

**Step 1**: Fetch Pfam annotations from UniProt REST API for each QfO species

**Step 2**: Construct ground-truth pairs (TP = shared Pfam domain + different architecture; TN = no shared domain)

Both steps are idempotent. Set `FORCE = True` to reprocess from scratch.

In [1]:
from pathlib import Path
import subprocess, sys

REPO_ROOT = Path("..")  # notebooks/ -> repo root
SCRIPTS_DIR = REPO_ROOT / "notebooks"  # the two build scripts this notebook drives
ANNOT_DIR = REPO_ROOT / "results/pfam_benchmark/annotations"
PAIRS_DIR = REPO_ROOT / "results/pfam_benchmark/pairs"

PYTHON = sys.executable
FORCE = False  # set True to reprocess all species
SPECIES = "all"  # or e.g. "mouse fly ecoli"

ANNOT_DIR.mkdir(parents=True, exist_ok=True)
PAIRS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python:    {PYTHON}")
print(f"Repo root: {REPO_ROOT.resolve()}")
print(f"Force:     {FORCE}")
print(f"Species:   {SPECIES}")

Python:    /Users/olga/anaconda3/envs/2025-kmerseek-analysis/bin/python
Repo root: /Users/olga/code/2024-kmerseek-analysis
Force:     False
Species:   all


## Step 1 - Fetch Pfam annotations from UniProt

Queries UniProt REST API for Pfam cross-references for each QfO species proteome.
~5-10 min for all 10 species (network-bound; 0.5 s polite delay between pages).

In [2]:
cmd = [
    PYTHON,
    str((SCRIPTS_DIR / "build_pfam_architectures.py").resolve()),
    "--species",
    SPECIES,
    "--outdir",
    str(ANNOT_DIR.resolve()),
]
if FORCE:
    cmd.append("--force")

print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=REPO_ROOT.resolve())
if proc.returncode != 0:
    raise RuntimeError(f"build_pfam_architectures.py failed (exit {proc.returncode})")

Running: /Users/olga/anaconda3/envs/2025-kmerseek-analysis/bin/python /Users/olga/code/2024-kmerseek-analysis/results/pfam_benchmark/build_pfam_architectures.py --species all --outdir /Users/olga/code/2024-kmerseek-analysis/results/pfam_benchmark/annotations
  human: already done — skipping (use --force to rerun)
  mouse: already done — skipping (use --force to rerun)
  chicken: already done — skipping (use --force to rerun)
  zebrafish: already done — skipping (use --force to rerun)
  ciona: already done — skipping (use --force to rerun)
  fly: already done — skipping (use --force to rerun)
  worm: already done — skipping (use --force to rerun)
  yeast: already done — skipping (use --force to rerun)
  arabidopsis: already done — skipping (use --force to rerun)
  ecoli: already done — skipping (use --force to rerun)

Done!


In [3]:
import polars as pl

domain_files = sorted(ANNOT_DIR.glob("*_pfam_domains.parquet"))
arch_files = sorted(ANNOT_DIR.glob("*_architectures.parquet"))
print(f"Domain parquets: {len(domain_files)}  |  Arch parquets: {len(arch_files)}")
print()

rows = []
for f in domain_files:
    sp = f.stem.replace("_pfam_domains", "")
    df = pl.read_parquet(f)
    rows.append(
        {
            "species": sp,
            "n_proteins": df["accession"].n_unique(),
            "n_domain_records": len(df),
        }
    )
print(pl.DataFrame(rows).sort("n_proteins", descending=True))

Domain parquets: 10  |  Arch parquets: 10

shape: (10, 3)
┌─────────────┬────────────┬──────────────────┐
│ species     ┆ n_proteins ┆ n_domain_records │
│ ---         ┆ ---        ┆ ---              │
│ str         ┆ i64        ┆ i64              │
╞═════════════╪════════════╪══════════════════╡
│ arabidopsis ┆ 22279      ┆ 34376            │
│ mouse       ┆ 20323      ┆ 35098            │
│ human       ┆ 18986      ┆ 33766            │
│ worm        ┆ 13659      ┆ 19352            │
│ zebrafish   ┆ 12525      ┆ 21589            │
│ fly         ┆ 10921      ┆ 17020            │
│ ciona       ┆ 10496      ┆ 15623            │
│ chicken     ┆ 6370       ┆ 11111            │
│ yeast       ┆ 5034       ┆ 7681             │
│ ecoli       ┆ 4109       ┆ 6150             │
└─────────────┴────────────┴──────────────────┘


## Step 2 - Construct ground-truth benchmark pairs

For each human x species pair:
- **Positive**: share >= 1 Pfam domain, different overall architecture, >= 1 protein has >= 2 domains
- **Negative**: share 0 Pfam domains

Targets ~20K positives + ~40K negatives per species (capped at 500 pairs/Pfam domain).

In [4]:
cmd = [
    PYTHON,
    str((SCRIPTS_DIR / "construct_subdomain_pairs.py").resolve()),
    "--species",
    SPECIES,
    "--annot-dir",
    str(ANNOT_DIR.resolve()),
    "--pairs-dir",
    str(PAIRS_DIR.resolve()),
]
if FORCE:
    cmd.append("--force")

print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=REPO_ROOT.resolve())
if proc.returncode != 0:
    raise RuntimeError(f"construct_subdomain_pairs.py failed (exit {proc.returncode})")

Running: /Users/olga/anaconda3/envs/2025-kmerseek-analysis/bin/python /Users/olga/code/2024-kmerseek-analysis/results/pfam_benchmark/construct_subdomain_pairs.py --species all --annot-dir /Users/olga/code/2024-kmerseek-analysis/results/pfam_benchmark/annotations --pairs-dir /Users/olga/code/2024-kmerseek-analysis/results/pfam_benchmark/pairs
  human_vs_mouse: already done — skipping (use --force)
  human_vs_chicken: already done — skipping (use --force)
  human_vs_zebrafish: already done — skipping (use --force)
  human_vs_ciona: already done — skipping (use --force)
  human_vs_fly: already done — skipping (use --force)
  human_vs_worm: already done — skipping (use --force)
  human_vs_yeast: already done — skipping (use --force)
  human_vs_arabidopsis: already done — skipping (use --force)
  human_vs_ecoli: already done — skipping (use --force)

Done!


In [5]:
pair_files = sorted(PAIRS_DIR.glob("human_vs_*_ground_truth.parquet"))
print(f"Ground truth parquets: {len(pair_files)}")
print()

rows = []
for f in pair_files:
    sp = f.stem.replace("human_vs_", "").replace("_ground_truth", "")
    df = pl.read_parquet(f)
    n_pos = int(df["label"].sum())
    rows.append(
        {"species": sp, "n_pos": n_pos, "n_neg": len(df) - n_pos, "total": len(df)}
    )
print(pl.DataFrame(rows).sort("total", descending=True))

Ground truth parquets: 9

shape: (9, 4)
┌─────────────┬───────┬───────┬───────┐
│ species     ┆ n_pos ┆ n_neg ┆ total │
│ ---         ┆ ---   ┆ ---   ┆ ---   │
│ str         ┆ i64   ┆ i64   ┆ i64   │
╞═════════════╪═══════╪═══════╪═══════╡
│ arabidopsis ┆ 20000 ┆ 40000 ┆ 60000 │
│ chicken     ┆ 20000 ┆ 40000 ┆ 60000 │
│ ciona       ┆ 20000 ┆ 40000 ┆ 60000 │
│ fly         ┆ 20000 ┆ 40000 ┆ 60000 │
│ mouse       ┆ 20000 ┆ 40000 ┆ 60000 │
│ worm        ┆ 20000 ┆ 40000 ┆ 60000 │
│ yeast       ┆ 20000 ┆ 40000 ┆ 60000 │
│ zebrafish   ┆ 20000 ┆ 40000 ┆ 60000 │
│ ecoli       ┆ 9803  ┆ 40000 ┆ 49803 │
└─────────────┴───────┴───────┴───────┘


## Summary

This is a data-build notebook, not an analysis, and both steps completed. It produces the ground
truth that [123](./123_pfam_subdomain_divergence_benchmark.ipynb) consumes.

Step 1 pulled Pfam annotations from UniProt for 10 QfO proteomes: 10 domain parquets and 10
architecture parquets, ranging from E. coli (4,109 proteins, 6,150 domain records) to Arabidopsis
(22,279 proteins, 34,376 records), with human at 18,986 proteins and 33,766 records.

Step 2 built 9 human-vs-species ground-truth pair sets. Eight hit the 20,000 positive / 40,000
negative target exactly, which means the cap bound rather than the data: positives are capped at 500
pairs per Pfam domain and then truncated at 20,000, so these sets are a sample of the available
positives, not all of them. E. coli is the exception at 9,803 positives, where the data ran out
before the cap. That asymmetry matters downstream, because E. coli is the most diverged species in
the panel and its smaller positive set will carry wider confidence intervals than the others.

Both steps are idempotent and every species reported "already done" on this run, so the outputs
above are a cache check rather than a fresh build. Set `FORCE = True` to rebuild from scratch if the
UniProt release or the species list changes.

Positives are defined as sharing at least one Pfam domain while having different overall
architectures, with at least one partner carrying 2+ domains; negatives share zero Pfam domains.
Proteins with no Pfam annotation at all are absent from both classes rather than counted as
negatives, so the benchmark is scoped to the annotated fraction of each proteome.